# Deduplicate Data (Demonstration)

Remove near-similar tweets

## A. Overview

Apply Reciprocal Rank Fusion (RRF) Hybrid seach using n-gram and all-MiniLM-L6-v2 Transformer model to filter out similar tweet

In [1]:
%pip install sentence-transformers
# install either faiss-cuda or faiss-cpu depending on your environment
# %pip install faiss-cuda
# %pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os

from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import numpy as np

import faiss

from sentence_transformers import SentenceTransformer
from huggingface_hub import login

import configuration
from src.normalizer import similarity

login(os.getenv("HF_TOKEN"))
dataset_path = Path('..') / 'data'

similarity_threshold = 0.8

/opt/homebrew/Caskroom/miniconda/base/envs/nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### Demonstration

#### The Lexical trap

In [3]:
# sample tweets for testing
df = pd.DataFrame({
    "tweet_text": [
        "Help! Trapped on the second floor! #flood",
        "The bird is trapped on the second floor.", # lexical similarity but not semantic
    ]})

In [4]:
similarity.filter_duplicates_with_resume(df, "tweet_text", similarity_threshold=similarity_threshold, chunk_size=len(df))

Original dataset size: 2
Vectorizing text...
Starting fresh chunked cosine similarity...
Processed up to row 2/2. Identified 0 near-duplicates in total.
Processing complete. Checkpoint file removed.
Final dataset size after near-duplicate removal: 2


,tweet_text
0,Help! Trapped on the second floor! #flood
1,The bird is trapped on the second floor.


In [5]:
# FAISS Semantic Setup (Using a lightweight pre-trained BERT model)
encoder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = encoder.encode(
    df['tweet_text'].astype(str).tolist()).astype('float32')

# Normalize embeddings for Cosine Similarity equivalence in FAISS L2 Search
faiss.normalize_L2(embeddings)
dimension = embeddings.shape[1]

# Initialize FAISS Index
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

query = "I am stucked upstairs. Need rescue from the flood!"
query_embedding = encoder.encode([query]).astype('float32')
faiss.normalize_L2(query_embedding)

# k-NN arbitrarily cuts off the flood reports
similarity.bruce_force_faiss_knn(df['tweet_text'].astype(str).tolist(), index, query_embedding, k=2)
# Radius captures all related spatial vectors.
similarity.bruce_force_faiss_radius(df['tweet_text'].astype(str).tolist(), index, query_embedding, similarity_threshold=similarity_threshold)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 25872.16it/s]



--- FAISS k-NN SEARCH (k=2) ---
[Similarity: 0.7292] Help! Trapped on the second floor! #flood
[Similarity: 0.2354] The bird is trapped on the second floor.

--- FAISS RADIUS SEARCH ---
[Similarity: 0.7292] Help! Trapped on the second floor! #flood
